<a href="https://colab.research.google.com/github/Nurdaylight/Study/blob/main/Text_scraper_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip -q install playwright
!python -m playwright install chromium
!python -m playwright install-deps chromium


Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,892 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,712 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiv

In [15]:
import asyncio
from playwright.async_api import async_playwright

START_URL = "https://novelbin.com/b/return-of-the-runebound-professor/chapter-200-panic"
BASE = "https://novelbin.com"

def clean_lines(text: str) -> str:
    lines = [ln.strip() for ln in text.splitlines()]
    return "\n".join([ln for ln in lines if ln])

async def scrape_n_chapters(start_url: str, n: int = 1, delay_sec: float = 1.0):
    async with async_playwright() as p:
        # These args fix most Colab/container crashes
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
                "--no-zygote",
                "--single-process",
            ],
        )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            locale="en-US",
        )
        page = await context.new_page()

        await page.set_extra_http_headers({
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://novelbin.com/",
        })

        url = start_url

        for i in range(n):
            print(f"\nScraping ({i+1}/{n}): {url}")

            # Use "networkidle" sometimes; if it hangs, switch back to "domcontentloaded"
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_selector("#chr-content", timeout=30000)

            title = (await page.locator("h2").first.inner_text()).strip()
            print("\n" + "=" * 80)
            print(title)
            print("=" * 80)

            raw = await page.locator("#chr-content").inner_text()
            print(clean_lines(raw))

            next_loc = page.locator("#next_chap")
            if await next_loc.count() == 0:
                print("\nNo next chapter link found. Stopping.")
                break

            href = await next_loc.get_attribute("href")
            if not href:
                print("\nNext chapter href missing. Stopping.")
                break

            url = href if href.startswith("http") else BASE + href
            await asyncio.sleep(delay_sec)

        await context.close()
        await browser.close()

# In Jupyter/Colab:
await scrape_n_chapters(START_URL, n=2, delay_sec=1.2)



Scraping (1/2): https://novelbin.com/b/return-of-the-runebound-professor/chapter-200-panic

Chapter 200: Panic
Chapter 200: Panic
Tenfort, Rank 6 mage and the strongest Enforcer in Arbitage, was panicking. He’d been a Rank 6 for forty-three years, and had just managed to form two Rank 6 Runes – three less than the headmaster, but still more than enough to put him on even footing with some of the most powerful mages in the Arbalest Empire.
He could even stand against some of the strongest family heads – but, against Ferdinand, he’d felt like he was a fly fighting a giant. Whoever the man was, he was at least as strong as the headmaster.
Shit. If I was alone, I’d get out of here and never come back, but I won’t abandon the students. The transport cannon doesn’t have any way for us to reset the timing on the recall, so we’re stuck here for the next few days. I won’t leave, but I just don’t know if there’s going to be much I can to do defend anyone against a monster like that – much less 

Error: Locator.get_attribute: Error: strict mode violation: locator("#next_chap") resolved to 2 elements:
    1) <a id="next_chap" class="btn btn-success" title="Chapter 201: Life" href="https://novelbin.com/b/return-of-the-runebound-professor/chapter-201-life">…</a> aka locator("#chr-nav-top").get_by_role("link", name="Next Chapter ")
    2) <a id="next_chap" class="btn btn-success" title="Chapter 201: Life" href="https://novelbin.com/b/return-of-the-runebound-professor/chapter-201-life">…</a> aka locator("#chr-nav-bottom").get_by_role("link", name="Next Chapter ")

Call log:
  - waiting for locator("#next_chap")


In [16]:
import asyncio
import re
from pathlib import Path
from playwright.async_api import async_playwright

START_URL = "https://novelbin.com/b/return-of-the-runebound-professor/chapter-200-panic"
OUT_DIR = Path("novelbin_chapters")
OUT_DIR.mkdir(exist_ok=True)

def slugify(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^\w\s-]", "", s)
    s = re.sub(r"[\s_-]+", "-", s)
    return s[:120] if s else "chapter"

def clean_text(text: str) -> str:
    lines = [ln.strip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln]
    return "\n".join(lines)

async def scrape_with_arrow_right(start_url: str, n: int = 5, delay_sec: float = 0.6):
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
                "--no-zygote",
                "--single-process",
            ],
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            locale="en-US",
        )
        page = await context.new_page()
        await page.goto(start_url, wait_until="domcontentloaded", timeout=60000)

        # Make sure the page is "focused" so keypresses work
        await page.click("body")

        for i in range(n):
            await page.wait_for_selector("#chr-content", timeout=30000)

            # scrape title + content
            title = (await page.locator("h2").first.inner_text()).strip()
            text = clean_text(await page.locator("#chr-content").inner_text())

            # save to txt
            filename = OUT_DIR / f"{i+1:04d}-{slugify(title)}.txt"
            filename.write_text(title + "\n\n" + text + "\n", encoding="utf-8")
            print(f"Saved: {filename}")

            # stop if last iteration
            if i == n - 1:
                break

            # capture state so we can detect the "next chapter" change
            old_url = page.url
            old_title = title

            # press right arrow to go next
            await page.keyboard.press("ArrowRight")

            # wait for navigation OR for title to change
            # (some sites use client-side routing, so URL might not change immediately)
            try:
                await page.wait_for_url(lambda url: url != old_url, timeout=15000)
            except:
                # fallback: wait until title changes
                await page.wait_for_function(
                    """(oldTitle) => {
                        const h2 = document.querySelector("h2");
                        return h2 && h2.innerText.trim() !== oldTitle;
                    }""",
                    arg=old_title,
                    timeout=15000
                )

            await asyncio.sleep(delay_sec)

        await context.close()
        await browser.close()

# Jupyter/Colab:
await scrape_with_arrow_right(START_URL, n=3, delay_sec=1.0)


Saved: novelbin_chapters/0001-chapter-200-panic.txt


TimeoutError: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("#chr-content") to be visible
